#Acquisition Notes

##Purpose:
This notebook acquires immutable raw snapshots of selected City of Hillsboro GIS datasets for the Web Hosted Portfolio project.

##Raw data convention:
data/raw/HIL-###_dataset_name/YYYY-MM-DD/HIL-###.json

Each snapshot is accompanied by manifest.json.

##Raw data policy:
No attribute or geometry transformations are performed during acquisition.

##Pagination:
ArcGIS REST services have dataset-specific transfer limits. The acquisition function retrieves records in batches until the complete layer has been downloaded.

##Versioning:
New acquisitions receive a new timestamped snapshot directory rather than overwriting previous snapshots.

##Storage:
Large raw JSON files are stored outside GitHub in the project's permanent raw-data storage. GitHub contains project code, documentation, and metadata/manifests.

##Known issue — HIL-001:
The public EcDev_SemiconductorBusinesses layer currently exposes metadata successfully but its /query endpoint returns Version 'sde.DEFAULT' is not accessible. This dataset remains unresolved pending source-side investigation.

##Acquisition date:
Initial collection performed August 25–26, 2026.

In [97]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import shutil
import requests

# --------------------------------------------------
# HIL Raw Data Acquisition
# --------------------------------------------------

SNAPSHOT_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

RAW_ROOT = Path(".")

print(f"Snapshot date: {SNAPSHOT_DATE}")
print(f"Raw data root: {RAW_ROOT}")

Snapshot date: 2026-08-26
Raw data root: .


In [98]:
DATASETS = {
    "HIL-001": {
        "name": "semiconductor_businesses",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "EcDev_SemiconductorBusinesses/FeatureServer/0"
        ),
        "source_organization": "City of Hillsboro"
    },

    "HIL-002": {
        "name": "project_boundaries",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "ContructionProjects/FeatureServer/0"
        ),
    },

    "HIL-003": {
        "name": "zoning",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "LandUseGallery_Prod/FeatureServer/23"
        ),
    },

    "HIL-004": {
        "name": "comprehensive_plan",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "LandUseGallery_Prod/FeatureServer/21"
        ),
    },

    "HIL-005": {
        "name": "buildings",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/91"
        ),
    },

    "HIL-006": {
        "name": "metro_buildings",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/92"
        ),
    },

    "HIL-007": {
        "name": "pavement_projects",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "ContructionProjects/FeatureServer/1"
        ),
    },

    "HIL-008": {
        "name": "city_limits",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Hillsboro_City_Limits/FeatureServer/0"
        ),
    },

    "HIL-009": {
        "name": "roadway",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/80"
        ),
    },
}

print(f"Datasets configured: {len(DATASETS)}")

Datasets configured: 9


In [99]:
def acquire_dataset(dataset_id, config):
    """
    Download a complete ArcGIS REST layer as raw JSON.

    - Retrieves records in batches.
    - Preserves source attributes and geometry.
    - Creates a timestamped snapshot directory.
    - Creates a manifest with metadata and SHA-256 checksum.
    """

    layer_url = config["url"].rstrip("/")
    query_url = f"{layer_url}/query"

    acquired_at = datetime.now(timezone.utc).isoformat()

    # --------------------------------------------------
    # Layer metadata
    # --------------------------------------------------

    layer_response = requests.get(
        layer_url,
        params={"f": "json"},
        timeout=60
    )

    layer_response.raise_for_status()

    layer_info = layer_response.json()

    if "error" in layer_info:
        raise RuntimeError(
            f"Layer metadata error for {dataset_id}: "
            f"{layer_info['error']}"
        )

    max_record_count = layer_info.get(
        "maxRecordCount",
        1000
    )

    print(f"Downloading {dataset_id}...")
    print(f"Query URL: {query_url}")
    print(f"Batch size: {max_record_count}")

    # --------------------------------------------------
    # Retrieve all records
    # --------------------------------------------------

    all_features = []
    offset = 0
    first_batch = None

    while True:

        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "resultOffset": offset,
            "resultRecordCount": max_record_count,
            "f": "json"
        }

        response = requests.get(
            query_url,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        batch = response.json()

        if first_batch is None:
            first_batch = batch

        if "error" in batch:
            raise RuntimeError(
                f"ArcGIS error for {dataset_id}: "
                f"{batch['error']}"
            )

        features = batch.get("features", [])

        if not features:
            break

        all_features.extend(features)

        print(
            f"  Retrieved {len(all_features):,} records..."
        )

        if not batch.get(
            "exceededTransferLimit",
            False
        ):
            break

        offset += len(features)

    # --------------------------------------------------
    # Validate acquisition
    # --------------------------------------------------

    record_count = len(all_features)

    if record_count == 0:
        raise RuntimeError(
            f"{dataset_id} returned zero records. "
            "No snapshot was saved."
        )

    # --------------------------------------------------
    # Reconstruct complete response
    # --------------------------------------------------

    data = {
        key: value
        for key, value in first_batch.items()
        if key != "features"
    }

    data["features"] = all_features

    # --------------------------------------------------
    # Create snapshot directories
    # --------------------------------------------------

    snapshot_dir = (
        RAW_ROOT /
        SNAPSHOT_DATE
    )

    datasets_dir = (
        snapshot_dir /
        "datasets"
    )

    manifests_dir = (
        snapshot_dir /
        "manifests"
    )

    datasets_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    manifests_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------
    # Save raw JSON
    # --------------------------------------------------

    data_file = (
        datasets_dir /
        f"{dataset_id}.json"
    )

    with open(
        data_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(data, f)

    # --------------------------------------------------
    # SHA-256 checksum
    # --------------------------------------------------

    sha256 = hashlib.sha256(
        data_file.read_bytes()
    ).hexdigest()

    # --------------------------------------------------
    # Manifest
    # --------------------------------------------------

    manifest = {
        "dataset_id": dataset_id,
        "dataset_name": config["name"],
        "source_organization": (
            config["source_organization"]
        ),
        "source_url": layer_url,
        "query_url": query_url,
        "acquired_at_utc": acquired_at,
        "snapshot_date": SNAPSHOT_DATE,
        "record_count": record_count,
        "file": data_file.name,
        "file_size_bytes": data_file.stat().st_size,
        "sha256": sha256,
        "notes": (
            "Raw acquisition snapshot. "
            "No attribute or geometry "
            "transformations performed."
        )
    }

    manifest_file = (
        manifests_dir /
        f"{dataset_id}_manifest.json"
    )

    with open(
        manifest_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            manifest,
            f,
            indent=2
        )

    print("\n✓ Complete acquisition")
    print(
        f"✓ Records: {record_count:,}"
    )
    print(
        f"✓ Saved: {data_file}"
    )
    print(
        f"✓ Manifest: {manifest_file}"
    )

    return manifest

In [100]:
# Used when downloading a single dataset
manifest = acquire_dataset(
    "HIL-002",
    DATASETS["HIL-002"]
)

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/ContructionProjects/FeatureServer/0/query
Batch size: 2000
  Retrieved 147 records...

✓ Complete acquisition
✓ Records: 147
✓ Saved: 2026-08-26/datasets/HIL-002.json
✓ Manifest: 2026-08-26/manifests/HIL-002_manifest.json


In [101]:
# datasets_to_acquire = [
#     "HIL-002",
#     "HIL-003",
#     "HIL-004",
#     "HIL-005",
#     "HIL-006",
#     "HIL-007",
#     "HIL-008",
#     "HIL-009",
# ]

# for dataset_id in datasets_to_acquire:
#     print("\n" + "=" * 70)
#     acquire_dataset(
#         dataset_id,
#         DATASETS[dataset_id]
#     )

In [102]:
print("HIL RAW DATA VALIDATION")
print("=" * 70)

for dataset_id, config in DATASETS.items():

    dataset_folder = (
        RAW_ROOT /
        f"{dataset_id}_{config['name']}"
    )

    if not dataset_folder.exists():
        print(
            f"⚠️ {dataset_id}: folder not found"
        )
        continue

    snapshots = sorted(
        dataset_folder.iterdir()
    )

    for snapshot_dir in snapshots:

        if not snapshot_dir.is_dir():
            continue

        json_file = (
            snapshot_dir /
            f"{dataset_id}.json"
        )

        manifest_file = (
            snapshot_dir /
            "manifest.json"
        )

        print(
            f"\n{dataset_id} — "
            f"{snapshot_dir.name}"
        )

        if not json_file.exists():
            print("  ❌ JSON file missing")
            continue

        if not manifest_file.exists():
            print("  ❌ Manifest missing")
            continue

        with open(
            manifest_file,
            "r",
            encoding="utf-8"
        ) as f:
            manifest = json.load(f)

        record_count = manifest.get(
            "record_count",
            0
        )

        if record_count == 0:
            print("  ❌ ZERO RECORDS")
        else:
            print(
                f"  ✓ Records: "
                f"{record_count:,}"
            )

        print(
            f"  ✓ JSON: "
            f"{json_file.stat().st_size:,} bytes"
        )

        print(
            f"  ✓ SHA-256: "
            f"{manifest.get('sha256', 'MISSING')[:16]}..."
        )

        print("  ✓ Manifest: present")

HIL RAW DATA VALIDATION
⚠️ HIL-001: folder not found
⚠️ HIL-002: folder not found
⚠️ HIL-003: folder not found
⚠️ HIL-004: folder not found
⚠️ HIL-005: folder not found
⚠️ HIL-006: folder not found
⚠️ HIL-007: folder not found
⚠️ HIL-008: folder not found
⚠️ HIL-009: folder not found


In [103]:
# Example: remove a known failed snapshot
#
# failed_snapshot = Path(
#     "data/HIL-001_semiconductor_businesses/2026-08-26"
# )
#
# if failed_snapshot.exists():
#     shutil.rmtree(failed_snapshot)
#     print("✓ Removed failed snapshot")

In [104]:
import shutil
from pathlib import Path

archive_path = shutil.make_archive(
    "/content/hillsboro_raw_2026-08-26",
    "zip",
    root_dir="/content",
    base_dir="data"
)

print(f"✓ Created: {archive_path}")

✓ Created: /content/hillsboro_raw_2026-08-26.zip


In [105]:
from google.colab import files

files.download(
    "/content/hillsboro_raw_2026-08-26.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [106]:
manifest = acquire_dataset(
    "HIL-001",
    DATASETS["HIL-001"]
)

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/EcDev_SemiconductorBusinesses/FeatureServer/0/query
Batch size: 2000
  Retrieved 59 records...

✓ Complete acquisition
✓ Records: 59
✓ Saved: 2026-08-26/datasets/HIL-001.json
✓ Manifest: 2026-08-26/manifests/HIL-001_manifest.json
